In [29]:
import os
import xarray as xr
import pandas as pd

# Set directory
#output_directory = "/Users/lb962/Documents/GitHub/ESL/data/processed_GESLA"

# Get list of all NetCDF files in the directory
#nc_files = [os.path.join(output_directory, f) for f in os.listdir(output_directory) if f.endswith('.nc')]

# Load and concatenate datasets along a new dimension if needed
#ds = xr.open_mfdataset(nc_files, combine='by_coords')  # or combine='nested' with concat_dim='time' if needed
ds = xr.open_dataset("/Users/lb962/Documents/GitHub/ESL/notebooks/1_data_prep/filtered_GESLA_sea_level.nc")
lat = ds['latitude'].values
lon = ds['longitude'].values

# If lat/lon are scalars or arrays, pair them
latlon_pairs = list(zip(lat.flatten(), lon.flatten()))
unique_latlon = list(set(latlon_pairs))  # remove duplicates

In [30]:
ds

<xarray.Dataset>
Dimensions:    (date_time: 657456, station: 188)
Coordinates:
  * date_time  (date_time) datetime64[ns] 1950-01-01 ... 2024-12-31T23:00:00
  * station    (station) int64 1 2 3 4 5 6 7 8 ... 420 421 422 423 424 425 426
    latitude   (station) float64 60.16 61.93 60.15 60.4 ... 58.43 59.0 58.01
    longitude  (station) float64 -1.14 5.117 -1.14 5.32 ... -3.083 9.856 7.555
Data variables:
    sea_level  (station, date_time) float32 ...

In [32]:
len(unique_latlon )

182

In [ ]:
import numpy as np
from shapely.geometry import Point
from shapely.ops import unary_union
import cartopy.io.shapereader as shpreader
from scipy.spatial import cKDTree

# Step 1: Generate 0.25° global grid (adjust bounds if needed)
lat_grid = np.arange(48, 64, 0.25)
lon_grid = np.arange(-15, 11, 0.25)
lat_mesh, lon_mesh = np.meshgrid(lat_grid, lon_grid, indexing='ij')
grid_points = np.column_stack([lat_mesh.ravel(), lon_mesh.ravel()])  # shape: (N, 2)

# Step 2: Load Natural Earth land polygons
reader = shpreader.natural_earth(resolution='10m', category='physical', name='land')
land_polygons = list(shpreader.Reader(reader).geometries())
land_geom = unary_union(land_polygons)

ocean_mask = [not land_geom.contains(Point(lon, lat)) for lat, lon in grid_points]
ocean_points = np.array(grid_points)[ocean_mask]

# Step 4: Build KDTree from ocean points for fast nearest-neighbor search
tree = cKDTree(ocean_points)

# Step 5: For each input lat/lon, find nearest ocean point
unique_latlon = list(set(unique_latlon))  # ensure uniqueness
input_coords = np.array(unique_latlon)
distances, indices = tree.query(input_coords)

# Step 6: Get the nearest ocean coordinates
snapped_to_ocean = ocean_points[indices]

# Step 7: Map back to input coordinates
matched = list(zip(map(tuple, input_coords), map(tuple, snapped_to_ocean)))

# ✅ Result: matched = [((original_lat, original_lon), (nearest_ocean_lat, nearest_ocean_lon)), ...]
print("Sample matched lat/lon to ocean grid:")
for orig, snap in matched[:5]:
    print(f"From {orig} → nearest ocean {snap}")


In [33]:
# Round each (lat, lon) pair to the nearest 0.25
rounded_latlon = [(round(lat * 4) / 4, round(lon * 4) / 4) for lat, lon in unique_latlon]

# Optional: Remove duplicates after rounding
rounded_latlon = list(set(rounded_latlon))

# Display sample
print("Sample rounded (lat, lon) pairs:", rounded_latlon)


Sample rounded (lat, lon) pairs: [(54.5, -0.5), (53.0, 5.0), (54.5, 9.75), (59.0, 9.75), (50.5, -4.75), (51.5, 3.5), (60.25, -1.25), (51.5, 3.75), (51.25, -4.0), (51.75, -5.0), (51.75, 4.25), (51.75, 4.0), (62.5, 6.25), (51.5, -4.0), (50.5, -2.0), (50.5, -2.5), (52.5, 4.5), (49.25, -2.0), (52.75, -4.0), (58.25, -6.5), (58.5, -5.0), (50.75, -1.0), (50.75, -2.75), (55.75, -6.25), (51.5, -2.75), (51.0, 2.25), (51.75, -10.0), (54.75, -5.0), (54.75, -5.75), (51.25, 3.75), (62.0, 5.0), (56.5, -6.0), (55.0, -1.5), (63.5, 9.0), (58.5, -3.0), (50.75, 0.0), (50.75, -0.5), (51.75, -8.0), (54.75, -3.5), (55.0, -8.5), (53.0, 4.75), (55.25, -7.25), (55.5, 9.75), (52.5, 1.75), (52.25, -6.5), (51.75, 3.75), (54.0, -4.75), (51.0, -1.5), (50.0, 1.0), (54.75, -8.5), (52.75, 4.75), (50.0, -6.25), (58.0, 7.5), (51.0, 1.75), (51.0, 1.25), (51.5, 0.75), (49.5, 0.0), (53.0, 1.25), (56.0, -3.25), (50.75, -1.25), (50.75, -1.75), (50.75, -1.5), (55.0, 10.0), (60.5, 5.25), (63.75, 8.75), (50.25, -4.25), (50.75, 1

In [23]:
import cdsapi
output_folder = "/Users/lb962/Documents/GitHub/ESL/data/ERA5_by_location"
for i, (lat, lon) in enumerate(rounded_latlon):
    out_file = os.path.join(output_folder, f"ERA5_{lat:.2f}_{lon:.2f}.zip")
    
    dataset = "reanalysis-era5-single-levels-timeseries"
    request = {
        "variable": [
            "2m_dewpoint_temperature",
            "2m_temperature",
            "total_precipitation",
            "mean_sea_level_pressure",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "surface_pressure",
            "sea_surface_temperature",
            "mean_wave_direction",
            "mean_wave_period",
            "significant_height_of_combined_wind_waves_and_swell"
        ],
        "location": {"longitude":  lon, "latitude": lat,},
        "date": ["1940-01-01/2025-07-30"],
        "data_format": "netcdf"
    }

    client = cdsapi.Client()
    client.retrieve(dataset, request).download(out_file)

2025-08-04 14:02:48,775 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2025-08-04 14:02:49,345 WARNING [2025-03-17T00:00:00] Please be aware that the generation of this dataset is using an alternative source for the ERA5 data and may be subject to changes over time (e.g. file format, data file structure, deprecation etc). This dataset should therefore be regarded as “experimental” and is **not recommended for use in a production environment**. 

Notification of changes via this catalogue entry banner and/or in the [Forum](https://forum.ecmwf.int/) will be provided on best efforts.
2025-08-04 14:02:49,346 INFO Request ID is f0ebc112-33dd-4cc0-9a22-4db10e7314b7
2025-08-04 14:02:49,513 INFO status has been updated to accepted
2025-08-04 14:03:03,761 INFO status has been updated to successful
2025-08-04 14:03:10,867 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and o